In [ ]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /home/ml-team/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/ml-team/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /home/ml-
[nltk_data]     team/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/ml-team/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## twitter

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from nltk.corpus import stopwords
from nltk import pos_tag, word_tokenize
from nltk.stem import WordNetLemmatizer # New import for lemmatization
from gensim.corpora import Dictionary
from gensim.models.phrases import Phrases, Phraser # New import for phrase modeling
import nltk
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
from sklearn.feature_extraction.text import CountVectorizer
from collections import defaultdict
import math # For log functions

# ---- NLTK resources (keep only valid ones) ----
nltk.download('punkt')
# nltk.download('punkt_tab') # This is not a valid NLTK package - kept as comment for historical context
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('wordnet') # *** NEW: Download for WordNetLemmatizer ***

# ==================================================================
# HYPERPARAMETERS (Adjusted for SOTA aspirations)
# ==================================================================
EMBED_DIM = 100
MAX_VOCAB = 3000
TOPIC_COUNT = 12
HIDDEN_DIM = 200
EPOCHS = 150 # *** INCREASED EPOCHS: More training for complex models ***
BATCH_SIZE = 128
LR = 0.0005
LAMBDA_COH = 8.0 # *** INCREASED: Stronger emphasis on coherence during training ***
LAMBDA_DIV = 3.0
LAMBDA_RED = 5.0
ROUTING_ITERS = 5
MAX_DOCS = 100000    # *** UPDATED: Capped at 2,000 docs as per pipeline title ***
BETA_KL = 0.05 # *** NEW: Weight for KL divergence (Beta-VAE concept) - Crucial for NMI ***
DROPOUT_RATE = 0.3 # *** NEW: Dropout rate for regularization in encoder ***

# ==================================================================
# PATHS (update as needed)
# ==================================================================

# DATA_PATH = 'E:/Presentation_journal_work/clean_data/tweets_dataset.txt'
DATA_PATH = "/content/drive/MyDrive/Topic Modelling All file/Final_Journal/dataset/clean_data/tweets_dataset.txt"

# DATA_PATH = "/content/drive/MyDrive/Topic Modelling All file/Final_Journal/dataset/clean_data/bbc_dataset.txt"
# DATA_PATH = "/content/drive/MyDrive/Topic Modelling All file/Final_Journal/dataset/clean_data/20newsgroups_dataset.txt"






# GLOVE_PATH = "E:/Presentation_journal_work/we/glove.6B.100d.txt"
GLOVE_PATH = "/content/drive/MyDrive/Topic Modelling All file/Final_Journal/embedding/glove.6B.100d.txt"

# ==================================================================
# BLOCK 1: ENHANCED DATA PREPROCESSING (capped at 2,000 docs)
# ==================================================================
print("Step 1: Enhanced Data Preprocessing")

def clean_and_filter_text(filepath):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer() # *** NEW: Initialize lemmatizer ***

    with open(filepath, 'r', encoding='utf-8') as f:
        raw_lines = f.readlines()
    tokenized_cleaned_docs = []
    for line in raw_lines:
        tokens = word_tokenize(line.strip().lower())
        tagged = pos_tag(tokens)
        filtered = [
            lemmatizer.lemmatize(word) # *** NEW: Apply lemmatization ***
            for word, tag in tagged
            if tag.startswith(('N', 'V', 'J', 'R')) # Noun, Verb, Adjective, Adverb
            and word.isalpha()
            and word not in stop_words
            and len(word) > 2
        ]
        if filtered:
            tokenized_cleaned_docs.append(filtered)
    return tokenized_cleaned_docs

docs_tokenized = clean_and_filter_text(DATA_PATH)

# --- Limit to first 2,000 documents if more are present ---
if len(docs_tokenized) > MAX_DOCS:
    docs_tokenized = docs_tokenized[:MAX_DOCS]

print(f"Total cleaned docs (capped): {len(docs_tokenized)}")

# *** NEW: Phrase Modeling (Bigrams) for better semantic units ***
print("Detecting and adding phrases (bigrams)...")
# min_count: Ignore all words and phrases with total collected count lower than this.
# threshold: Represent a threshold for forming the phrases (higher means fewer phrases).
phrases = Phrases(docs_tokenized, min_count=5, threshold=10)
bigram_phraser = Phraser(phrases)
docs_tokenized_phrased = [bigram_phraser[doc] for doc in docs_tokenized]
# print(f"Example of original doc tokens: {docs_tokenized[0] if docs_tokenized else 'N/A'}")
# print(f"Example of phrased doc tokens: {docs_tokenized_phrased[0] if docs_tokenized_phrased else 'N/A'}")


# Create dictionary with stricter filtering - *** NOW USES phrased docs ***
dictionary = Dictionary(docs_tokenized_phrased)
dictionary.filter_extremes(no_below=10, no_above=0.3, keep_n=MAX_VOCAB)
dictionary.compactify()
VOCAB_SIZE = len(dictionary)
i2w = {v: k for k, v in dictionary.token2id.items()}

# Build BoW tensor - *** NOW USES phrased docs ***
docs_joined = [" ".join(doc) for doc in docs_tokenized_phrased]
vectorizer = CountVectorizer(vocabulary=dictionary.token2id)  # fixed vocab
X_bow = vectorizer.fit_transform(docs_joined).toarray()
bow_tensor = torch.tensor(X_bow, dtype=torch.float32)

print(f"Dictionary size: {VOCAB_SIZE}")
print("Step 1 complete.\n" + "-"*70)

# ==================================================================
# BLOCK 2: EMBEDDINGS & PMI MATRIX, AND COHERENCE METRICS DEFINITION (Moved here for scope)
# ==================================================================
print("Step 2: Embeddings & PMI Matrix")

def compute_pmi_matrix(docs_tokenized, vocab_size, dictionary):
    cooc = np.zeros((vocab_size, vocab_size), dtype=np.float64)
    window_size = 10
    for doc in docs_tokenized: # *** USES phrased docs for PMI calculation ***
        doc_ids = [dictionary.token2id[w] for w in doc if w in dictionary.token2id]
        for i, wi in enumerate(doc_ids):
            start = max(0, i - window_size)
            end = min(len(doc_ids), i + window_size + 1)
            context = doc_ids[start:i] + doc_ids[i+1:end]
            for wj in context:
                cooc[wi, wj] += 1.0
    total_pairs = cooc.sum()
    if total_pairs == 0:
        total_pairs = 1.0
    p_i = cooc.sum(axis=1) / total_pairs
    p_j = cooc.sum(axis=0) / total_pairs
    p_ij = cooc / total_pairs
    pmi = np.log((p_ij + 1e-10) / (p_i[:, None] * p_j[None, :] + 1e-10))
    return torch.tensor(pmi, dtype=torch.float32)

pmi_matrix = compute_pmi_matrix(docs_tokenized_phrased, VOCAB_SIZE, dictionary) # *** USES phrased docs ***
print(f"PMI matrix shape: {pmi_matrix.shape}")

def load_glove_embeddings(path, vocab, dim=100):
    embedding_dict = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split()
            if not parts:
                continue
            word = parts[0]
            # Handle potential phrases in GloVe (e.g., "new_york" might not be there, "new york" might)
            # This is a simplified lookup, for robust phrase handling, one might avg embeddings.
            if word in vocab and len(parts) == dim + 1:
                vector = np.asarray(parts[1:], dtype=np.float32)
                embedding_dict[word] = vector

    embedding_matrix = np.random.normal(0, 0.01, (len(vocab), dim)).astype(np.float32)
    for word, idx in vocab.items():
        if word in embedding_dict:
            embedding_matrix[idx] = embedding_dict[word]
        # For multi-word phrases, if not found in GloVe as one token,
        # consider averaging individual word embeddings (more complex, but better).
        # For simplicity here, if a phrase isn't in glove, it gets random init.
        elif '_' in word: # Handle multi-word phrases from gensim.phrases
            individual_words = word.split('_')
            phrase_vec = np.zeros(dim, dtype=np.float32)
            word_count = 0
            for w in individual_words:
                if w in embedding_dict:
                    phrase_vec += embedding_dict[w]
                    word_count += 1
            if word_count > 0:
                embedding_matrix[idx] = phrase_vec / word_count
            # Else, it remains random normal init.

    return torch.tensor(embedding_matrix, dtype=torch.float32)

word_embeddings = load_glove_embeddings(GLOVE_PATH, dictionary.token2id, dim=EMBED_DIM)
print(f"Embedding matrix shape: {word_embeddings.shape}")

# *** Pre-compute document frequencies and co-document frequencies for coherence metrics ***
# This section is crucial and must be defined before training loop.
print("Pre-computing document frequencies for coherence metrics...")
def _get_df_dcf_refined(docs_tokenized, dictionary):
    df = defaultdict(int) # document frequency (word_id -> count)
    dcf = defaultdict(lambda: defaultdict(int)) # co-document frequency (word_id1 -> word_id2 -> count)

    for doc_tokens in docs_tokenized: # *** USES phrased docs ***
        doc_word_ids = set(dictionary.token2id[word] for word in doc_tokens if word in dictionary.token2id)

        for word_id in doc_word_ids:
            df[word_id] += 1

        sorted_doc_word_ids = sorted(list(doc_word_ids))
        for i, word1_id in enumerate(sorted_doc_word_ids):
            for j in range(i + 1, len(sorted_doc_word_ids)):
                word2_id = sorted_doc_word_ids[j]
                dcf[word1_id][word2_id] += 1
                dcf[word2_id][word1_id] += 1 # Ensure symmetry

    return df, dcf

df, dcf = _get_df_dcf_refined(docs_tokenized_phrased, dictionary) # *** USES phrased docs ***
N_docs = len(docs_tokenized_phrased) # *** USES phrased docs length ***
print("Document frequencies computed.")


# ==================================================================
# *** Custom Coherence Metrics Implementation (MOVED HERE) ***
# ==================================================================

# Helper for NPMI used in c_npmi and c_v
def _calculate_pair_npmi_doc_level(w1_id, w2_id, df_dict, dcf_dict, num_docs, epsilon=1e-10):
    if w1_id == w2_id: return 1.0

    w1_id, w2_id = min(w1_id, w2_id), max(w1_id, w2_id)

    cnt_ab = dcf_dict[w1_id][w2_id]
    cnt_a = df_dict[w1_id]
    cnt_b = df_dict[w2_id]

    p_ab = (cnt_ab + epsilon) / num_docs
    p_a = (cnt_a + epsilon) / num_docs
    p_b = (cnt_b + epsilon) / num_docs

    if p_a <= 0 or p_b <= 0 or p_ab <= 0:
        return -1.0 # Default to -1.0 if not co-occurring or very low probability

    pmi = math.log(p_ab / (p_a * p_b))
    # denom = -math.log(p_ab)
    denom = math.log(p_ab)

    return pmi / denom if denom != 0.0 else 0.0

def compute_coherence_npmi(topics, docs_tokenized, dictionary, top_n=10):
    total_coherence = 0.0
    num_pairs = 0

    for topic in topics:
        topic_word_ids = [dictionary.token2id[w] for w in topic if w in dictionary.token2id]
        if len(topic_word_ids) < 2: continue

        for i in range(len(topic_word_ids)):
            for j in range(i + 1, len(topic_word_ids)):
                w1_id = topic_word_ids[i]
                w2_id = topic_word_ids[j]

                if w1_id not in dictionary.token2id.values() or w2_id not in dictionary.token2id.values():
                    continue

                npmi_val = _calculate_pair_npmi_doc_level(w1_id, w2_id, df, dcf, N_docs)
                total_coherence += npmi_val
                num_pairs += 1

    return total_coherence / num_pairs if num_pairs > 0 else -1.0

def compute_coherence_umass(topics, docs_tokenized, dictionary, epsilon=1.0, top_n=10):
    total_coherence = 0.0
    num_pairs = 0

    for topic in topics:
        topic_word_ids = [dictionary.token2id[w] for w in topic if w in dictionary.token2id]
        if len(topic_word_ids) < 2: continue

        for i in range(len(topic_word_ids)):
            w_i_id = topic_word_ids[i]

            if df[w_i_id] == 0: continue

            for j in range(len(topic_word_ids)):
                if i == j: continue
                w_j_id = topic_word_ids[j]

                if w_j_id not in dictionary.token2id.values():
                    continue

                d_wjw_i_id1 = min(w_j_id, w_i_id)
                d_wjw_i_id2 = max(w_j_id, w_i_id)

                d_wjw_i = dcf[d_wjw_i_id1][d_wjw_i_id2]
                d_wi = df[w_i_id]

                if d_wi == 0:
                    coherence_val = -float('inf')
                else:
                    coherence_val = math.log((d_wjw_i + epsilon) / d_wi)

                total_coherence += coherence_val
                num_pairs += 1

    return total_coherence / num_pairs if num_pairs > 0 else float('inf')
    # return total_coherence / num_pairs if num_pairs > 0 else -float('inf')

def compute_coherence_uci(topics, docs_tokenized, dictionary, epsilon=1.0, top_n=10):
    total_coherence = 0.0
    num_pairs = 0

    for topic in topics:
        topic_word_ids = [dictionary.token2id[w] for w in topic if w in dictionary.token2id]
        if len(topic_word_ids) < 2: continue

        for i in range(len(topic_word_ids)):
            for j in range(i + 1, len(topic_word_ids)):
                w1_id = topic_word_ids[i]
                w2_id = topic_word_ids[j]

                if w1_id not in dictionary.token2id.values() or w2_id not in dictionary.token2id.values():
                    continue

                d_w1w2_id1 = min(w1_id, w2_id)
                d_w1w2_id2 = max(w1_id, w2_id)

                d_w1w2 = dcf[d_w1w2_id1][d_w1w2_id2]
                d_w1 = df[w1_id]
                d_w2 = df[w2_id]

                numerator = d_w1w2 + epsilon
                denominator = (d_w1 * d_w2) + epsilon

                if denominator == 0:
                    coherence_val = -float('inf')
                else:
                    coherence_val = math.log(numerator / denominator)

                total_coherence += coherence_val
                num_pairs += 1

    return total_coherence / num_pairs if num_pairs > 0 else float('inf')
    # return total_coherence / num_pairs if num_pairs > 0 else -float('inf')

def compute_coherence_cv(topics, docs_tokenized, dictionary, word_embeddings, epsilon=1e-10, top_n=10):
    total_cv_score = 0.0
    num_topics_for_cv = 0

    for topic in topics:
        topic_word_ids = [dictionary.token2id[w] for w in topic if w in dictionary.token2id]
        if len(topic_word_ids) < 2: continue

        segment_vectors = {}

        for i, w_i_id in enumerate(topic_word_ids):
            if w_i_id not in dictionary.token2id.values(): continue

            temp_segment_vector = np.zeros(word_embeddings.shape[1], dtype=np.float32)

            for j, w_j_id in enumerate(topic_word_ids):
                if i == j: continue
                if w_j_id not in dictionary.token2id.values(): continue

                npmi_val = _calculate_pair_npmi_doc_level(w_i_id, w_j_id, df, dcf, N_docs, epsilon)

                if npmi_val > 0: # Only add if NPMI is positive (words are associated)
                    temp_segment_vector += npmi_val * word_embeddings[w_j_id].cpu().numpy()

            if np.linalg.norm(temp_segment_vector) > 1e-6:
                segment_vectors[w_i_id] = temp_segment_vector

        topic_pair_cv_scores = []
        seg_vec_ids = list(segment_vectors.keys())

        if len(seg_vec_ids) < 2: continue

        for i in range(len(seg_vec_ids)):
            for j in range(i + 1, len(seg_vec_ids)):
                s1 = segment_vectors[seg_vec_ids[i]]
                s2 = segment_vectors[seg_vec_ids[j]]

                norm_s1 = np.linalg.norm(s1)
                norm_s2 = np.linalg.norm(s2)

                if norm_s1 > 1e-6 and norm_s2 > 1e-6:
                    similarity = np.dot(s1, s2) / (norm_s1 * norm_s2)
                    topic_pair_cv_scores.append(similarity)

        if topic_pair_cv_scores:
            total_cv_score += np.mean(topic_pair_cv_scores)
            num_topics_for_cv += 1

    return total_cv_score / num_topics_for_cv if num_topics_for_cv > 0 else 0.0

print("Step 2 complete.\n" + "-"*70)


# ==================================================================
# BLOCK 3: ADVANCED MODEL ARCHITECTURE
# ==================================================================
print("Step 3: Advanced Model Architecture")

class CapsuleRouting(nn.Module):
    def __init__(self, input_dim, topic_dim, routing_iters=ROUTING_ITERS):
        super().__init__()
        self.W = nn.Parameter(torch.randn(topic_dim, input_dim))
        self.routing_iters = routing_iters
        self.input_dim = input_dim
        self.topic_dim = topic_dim

    def forward(self, x):
        batch_size = x.size(0)
        b = torch.zeros(batch_size, self.topic_dim, device=x.device)
        for _ in range(self.routing_iters):
            c = F.softmax(b, dim=1)
            s = torch.matmul(x, self.W.t())
            v = torch.tanh(s)
            agreement = (s * v).sum(dim=1, keepdim=True)
            b = b + agreement
        return F.softmax(v, dim=1)

class TopicEncoder(nn.Module):
    # *** NEW: Added dropout_rate parameter ***
    def __init__(self, vocab_size, hidden_dim, topic_dim, dropout_rate=0.0):
        super().__init__()
        self.fc1 = nn.Linear(vocab_size, hidden_dim)
        self.dropout1 = nn.Dropout(dropout_rate) # *** NEW: Dropout layer ***

        # Optional: Add another layer for more capacity if needed
        # self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        # self.dropout2 = nn.Dropout(dropout_rate)

        self.fc_mu = nn.Linear(hidden_dim, topic_dim)
        self.fc_logvar = nn.Linear(hidden_dim, topic_dim)
        self.capsule = CapsuleRouting(input_dim=topic_dim, topic_dim=topic_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.dropout1(h) # *** NEW: Apply dropout ***

        # Optional:
        # h = F.relu(self.fc2(h))
        # h = self.dropout2(h)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        theta = self.capsule(z)
        return z, theta, mu, logvar

class TopicDecoder(nn.Module):
    def __init__(self, K, word_embeds, embed_dim, rho_init=1.5):
        super().__init__()
        self.K = K
        self.embed_dim = embed_dim
        self.word_embeds = word_embeds
        self.V = word_embeds.shape[0]
        self.topic_vectors = nn.Parameter(torch.randn(K, embed_dim))
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.alpha = nn.Parameter(torch.ones(K, 1))

    def forward(self):
        dot_product = torch.matmul(self.topic_vectors, self.word_embeds.T)
        smoothed = self.rho * dot_product + self.alpha
        beta = F.softmax(smoothed, dim=1)
        return beta

def CRT_counts(x_bow, theta, beta_matrix):
    with torch.no_grad():
        batch_size, V = x_bow.shape
        K = theta.shape[1]
        Npz = torch.zeros(batch_size, K)
        Npw = torch.zeros(K, V)
        for i in range(batch_size):
            word_counts = x_bow[i]
            for j in range(V):
                count = int(word_counts[j].item())
                if count > 0:
                    probs = theta[i] * beta_matrix[:, j]
                    if probs.sum() == 0:
                        continue
                    probs = probs / probs.sum()
                    topic_samples = torch.multinomial(probs, count, replacement=True)
                    for k in topic_samples:
                        Npz[i, k] += 1
                        Npw[k, j] += 1
        return Npz, Npw

class CoherenceAwareLoss(nn.Module):
    def __init__(self, pmi_matrix, weight=LAMBDA_COH):
        super().__init__()
        self.pmi_matrix = pmi_matrix
        self.weight = weight

    def forward(self, beta):
        top_words = torch.topk(beta, 10, dim=1).indices
        coherence_loss = 0
        for k in range(beta.shape[0]):
            words = top_words[k]
            for i in range(len(words)):
                for j in range(i+1, len(words)):
                    wi = words[i].item()
                    wj = words[j].item()
                    # We want to maximize PMI, so we minimize -PMI
                    coherence_loss -= self.pmi_matrix[wi, wj]
        return self.weight * coherence_loss / beta.shape[0]

def topic_diversity_loss(beta):
    # Train-time diversity penalty (avg pairwise cosine similarity off-diagonal)
    norm_beta = F.normalize(beta, p=2, dim=1)
    sim = torch.mm(norm_beta, norm_beta.t())
    diversity_penalty = (sim.sum() - sim.trace()) / (beta.shape[0] * (beta.shape[0] - 1) + 1e-10)
    return LAMBDA_DIV * diversity_penalty

def redundancy_penalty(beta, top_n=10):
    top_words = torch.topk(beta, top_n, dim=1).indices.cpu().numpy()
    word_counts = {}
    for topic in top_words:
        for w in topic:
            word_counts[w] = word_counts.get(w, 0) + 1
    penalty = sum((count - 1) for count in word_counts.values() if count > 1)
    return LAMBDA_RED * penalty / (beta.shape[0] * top_n + 1e-10)

print("Advanced models initialized")
print("Step 3 complete.\n" + "-"*70)

# ==================================================================
# BLOCK 4: TRAINING
# ==================================================================
print("Step 4: Optimized Training")

def reconstruct_x(theta, beta):
    x_hat = torch.matmul(theta, beta)
    x_recon_log = torch.log(x_hat + 1e-10)
    return x_recon_log

# *** MODIFIED: elbo_loss now includes BETA_KL weight ***
def elbo_loss(x, x_recon_log, mu, logvar, beta_kl_weight=1.0):
    recon_loss = -torch.sum(x * x_recon_log, dim=1).mean()
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    return recon_loss + beta_kl_weight * kl_div # Apply beta_kl_weight here

def compute_total_loss(x, x_recon_log, mu, logvar, beta, coh_loss_fn):
    loss = elbo_loss(x, x_recon_log, mu, logvar, BETA_KL) # *** PASS BETA_KL ***
    loss += coh_loss_fn(beta)
    loss += topic_diversity_loss(beta)
    loss += redundancy_penalty(beta, top_n=10)
    return loss

encoder = TopicEncoder(VOCAB_SIZE, HIDDEN_DIM, TOPIC_COUNT, dropout_rate=DROPOUT_RATE) # *** PASS dropout_rate ***
decoder = TopicDecoder(TOPIC_COUNT, word_embeddings, EMBED_DIM)
coh_loss_fn = CoherenceAwareLoss(pmi_matrix)

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=LR
)

dataset = TensorDataset(bow_tensor)
train_dl = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)


for epoch in range(1, EPOCHS+1):
    encoder.train()
    decoder.train()
    running_loss = 0.0

    for batch in train_dl:
        x = batch[0]
        optimizer.zero_grad()
        z, theta, mu, logvar = encoder(x)
        beta = decoder()
        _Npz, _Npw = CRT_counts(x, theta, beta)
        x_recon_log = reconstruct_x(theta, beta)
        loss = compute_total_loss(x, x_recon_log, mu, logvar, beta, coh_loss_fn)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_dl)
    if epoch % 10 == 0: # Check less frequently now that custom metrics are computed for training progress
        # print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f}")
        # Quick coherence check during training (using custom NPMI, which is now defined)
        with torch.no_grad():
            beta_val = decoder()
            top_words = [torch.topk(beta_val[k], 10)[1].tolist() for k in range(TOPIC_COUNT)]
            topics_quick = [[i2w[i] for i in ids] for ids in top_words]

            try:
                # *** IMPORTANT: Use docs_tokenized_phrased here for coherence calculations ***
                npmi_quick = compute_coherence_npmi(topics_quick, docs_tokenized_phrased, dictionary, top_n=10)
                print(f"  Current NPMI: {npmi_quick:.4f}")
            except Exception as e:
                print(f"  Error calculating custom NPMI during quick check: {e}")

print("Step 4 complete.\n" + "-"*70)

# ==================================================================
# BLOCK 5: POST-TRAINING & EVALUATION
# ==================================================================
print("Step 5: Post-Training Processing & Evaluation")

def get_top_words(beta, i2w, top_n=10):
    topics = []
    for k in range(beta.shape[0]):
        top_ids = torch.topk(beta[k], top_n)[1].tolist()
        top_words = [i2w[i] for i in top_ids]
        topics.append(top_words)
    return topics

beta_final = decoder().detach()

def topic_diversity_cosine(beta: torch.Tensor) -> float:
    if not isinstance(beta, torch.Tensor):
        beta = torch.tensor(beta, dtype=torch.float32)
    K = beta.size(0)
    if K < 2:
        return 1.0
    Bn = F.normalize(beta, p=2, dim=1)
    S = torch.matmul(Bn, Bn.t())
    iu = torch.triu_indices(K, K, offset=1)
    if iu.numel() == 0:
        return 1.0
    avg_sim = S[iu[0], iu[1]].mean().item()
    return 1.0 - float(avg_sim)

topic_diversity_score = topic_diversity_cosine(beta_final)
print(f"Topic Diversity: {topic_diversity_score:.4f}")

topics = get_top_words(beta_final, i2w, top_n=10)

def filter_redundant_topics(topics, similarity_threshold=0.7):
    topic_sets = [set(topic) for topic in topics]
    unique_topics = []
    unique_topic_indices = []

    for i, topic in enumerate(topic_sets):
        redundant = False
        for j, other_topic_idx in enumerate(unique_topic_indices):
            other_topic_set = topic_sets[other_topic_idx]
            overlap = len(topic.intersection(other_topic_set))
            min_len = min(len(topic), len(other_topic_set))

            if min_len > 0 and overlap / min_len > similarity_threshold:
                redundant = True
                break
        if not redundant:
            unique_topics.append(topics[i])
            unique_topic_indices.append(i)
    return unique_topics

filtered_topics = filter_redundant_topics(topics)
print(f"\nOriginal topics: {len(topics)}, Filtered topics: {len(filtered_topics)}")

print("\nTop Words Per Topic (Filtered):")
for i, topic in enumerate(filtered_topics):
    print(f"Topic {i+1}: {', '.join(topic)}")

# Calculate all custom coherence scores - *** IMPORTANT: Use docs_tokenized_phrased here ***
npmi_score = compute_coherence_npmi(filtered_topics, docs_tokenized_phrased, dictionary)
uci_score = compute_coherence_uci(filtered_topics, docs_tokenized_phrased, dictionary)
umass_score = compute_coherence_umass(filtered_topics, docs_tokenized_phrased, dictionary)
cv_score = compute_coherence_cv(filtered_topics, docs_tokenized_phrased, dictionary, word_embeddings)

print(f"\nCoherence (c_v): {cv_score:.4f}")
print(f"Coherence (c_npmi): {npmi_score:.4f}")
print(f"Coherence (c_uci): {uci_score:.4f}")
print(f"Coherence (c_umass): {umass_score:.4f}")


# NMI w.r.t. argmax-topic pseudo-labels
theta_np = []
encoder.eval()
with torch.no_grad():
    for batch in train_dl:
        z, theta, mu, logvar = encoder(batch[0])
        theta_np.append(theta.cpu().numpy())
theta_np = np.vstack(theta_np)

kmeans = KMeans(n_clusters=TOPIC_COUNT, n_init=10, random_state=42)
clusters = kmeans.fit_predict(theta_np)
true_topics = np.argmax(theta_np, axis=1)
nmi_score = normalized_mutual_info_score(clusters, true_topics)
print(f"NMI Score: {nmi_score:.4f}")

print("\nStep 5 complete.\n" + "-"*70)

# ==================================================================
# SAVE FINAL MODEL & TOPICS
# ==================================================================
torch.save({
    'encoder': encoder.state_dict(),
    'decoder': decoder.state_dict(),
    'topics': filtered_topics,
    'beta': beta_final
}, 'topic_model_final.pth')
print("✅ Model saved to 'topic_model_final.pth'")


[nltk_data] Downloading package punkt to /home/ml-team/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/ml-team/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/ml-team/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /home/ml-
[nltk_data]     team/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/ml-team/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Step 1: Enhanced Data Preprocessing
Total cleaned docs (capped): 100000
Detecting and adding phrases (bigrams)...
Dictionary size: 1791
Step 1 complete.
----------------------------------------------------------------------
Step 2: Embeddings & PMI Matrix
PMI matrix shape: torch.Size([1791, 1791])
Embedding matrix shape: torch.Size([1791, 100])
Pre-computing document frequencies for coherence metrics...
Document frequencies computed.
Step 2 complete.
----------------------------------------------------------------------
Step 3: Advanced Model Architecture
Advanced models initialized
Step 3 complete.
----------------------------------------------------------------------
Step 4: Optimized Training
  Current NPMI: 0.5082
  Current NPMI: 0.5147
  Current NPMI: 0.5093
  Current NPMI: 0.5121
  Current NPMI: 0.5150
  Current NPMI: 0.5203
  Current NPMI: 0.5155
  Current NPMI: 0.5186
  Current NPMI: 0.5222
  Current NPMI: 0.5255
  Current NPMI: 0.5256
  Current NPMI: 0.5217
  Current NPMI: 0.5